# Aşama 2 baseline: TR-Omni-E4B ve emotion2vec+ (eğitimsiz)

İnce ayardan önce çıtayı ölçer. Eğitim yok; modeller hazır hâliyle çalışır.

| Bölüm | Veri | Ne ölçülür |
|---|---|---|
| `e2v_algi` | TurEV-DB, sınıf başına 40 kayıt (160) | emotion2vec+ base hazır etiketi: 4 duygu UAR, "kızgın mı?" UAR |
| `omni_algi` | Aynı 160 kayıt | TR-Omni'ye sesle birlikte "duygu hangisi?" sorulur: aynı ölçütler + tanınamayan cevap |
| `omni_ofke` | 20 ElevenLabs müşteri tepkisi | TR-Omni'nin 0-4 öfke puanı ile mevcut hattın ölçtüğü öfkenin sıra korelasyonu |
| `omni_sohbet` | Aynı 20 tepki | TR-Omni sesli cevap: süre, Türkçe yazım denetimi, cevap sesinin duygusu; sesler dinlenebilir |
| `omni_gecikme` | Aynı 20 tepki | Akışlı kullanımda ilk ses süresi: düşünürün ilk cümlesi + konuşucunun o cümleyi seslendirmesi; GPU adıyla ayrı kaydedilir |
| `omni_enjeksiyon` | Aynı 20 tepki × 3 koşul | Öfke bilgisi metin olarak verilince (yok / sakin / öfkeli) cevap ve modelin seçtiği ton değişiyor mu? |
| `omni_kalite` | `omni_gecikme` cevaplarının bütün cümleleri × 5 ayar | Konuşucunun ses kalitesi: Whisper ile yazıya geri çevirip metinle karakter hata oranı (CER); bozuk cümle oranı |
| `omni_mos` | `omni_kalite` ve `omni_konusucu` sesleri | Tahmini dinleyici puanı (UTMOS, 1-5): cızırtı, metalik ses gibi bozulmalar |
| `omni_konusucu` | `omni_gecikme` cevapları × 6 ayar | Konuşucu hızlandırma: akışlı çözme, + derleme, difüzyon adımı 16/8/6/4, arka planda düşünür çalışırken. İlk ses parçası, gerçek zaman oranı |

**Çalışma zamanı:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **L4** veya **A100**. TR-Omni ~17 GB GPU belleği
ister; T4 (16 GB) yetmez. Model indirmesi ~21 GB, ilk çağrıda ~1 dk derleme.

**Drive hazırlığı (bir kez):** `MyDrive/DuyguAvcisi/enjeksiyon_ses/` klasörüne yerel bilgisayardaki
`data/enjeksiyon_ses/` içeriğini (20 `.wav` + `sonuc.jsonl`) yükleyin. Bu sesler git'e girmiyor; ElevenLabs ile üretilmiş
sentetik seslerdir, kişisel veri içermez. Sonuçlar `MyDrive/DuyguAvcisi/s2s_baseline/` altına yazılır; Colab koparsa
hücreyi yeniden çalıştırın, kaldığı yerden devam eder.

In [ ]:
# 1) Drive, kod ve veri
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/DuyguAvcisi"
REPO = "https://github.com/Pappates/Emotional_Feedback_Costumer_Service"
DAL = "asama2/s2s-baseline"
# Depo özelse: from getpass import getpass; REPO = REPO.replace("https://", f"https://{getpass('GitHub token: ')}@")
if not os.path.exists("proje"):
    !git clone -q -b {DAL} {REPO} proje
else:
    !cd proje && git pull -q
if not os.path.exists("turev"):
    !git clone -q --depth 1 https://github.com/Xeonen/TurEV-DB turev
MUSTERI = f"{DRIVE}/enjeksiyon_ses"
print("Müşteri sesleri:", len([f for f in os.listdir(MUSTERI) if f.endswith(".wav")]) if os.path.isdir(MUSTERI) else "YOK, Drive'a yükleyin")

In [ ]:
# 2) Paketler (TR-Omni: transformers>=5.17, voxcpm==2.0.3, PyTorch 2.8+)
!pip -q install "transformers>=5.17" voxcpm==2.0.3 funasr librosa soundfile zeyrek wordfreq
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
if torch.cuda.is_available():
    print(f"GPU belleği: {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB (en az ~17 GB gerekli)")

In [ ]:
# 3) TR-Omni-E4B indir (~21 GB, Apache-2.0)
from huggingface_hub import snapshot_download
MODEL = snapshot_download("furkanbekmezci/TR-Omni-E4B-Turkish-Native-Speech-to-Speech-Model", local_dir="/content/tr_omni")
print(MODEL)

In [ ]:
# 4) Ölçüm. Biten bölümler atlanır; omni_mos birkaç dk (yalnızca kayıtlı sesleri puanlar). T4'e sığmaz; L4 veya A100 kullanın. Yalnızca bir bölüm için: --bolum omni_konusucu
CIKTI = f"{DRIVE}/s2s_baseline"
!cd proje && python egitim/s2s_baseline.py --turev "/content/turev/Sound Source" --model {MODEL} --musteri {MUSTERI} --cikti {CIKTI} --adet 40 2>&1 | grep -v "Warning"

## Nasıl okunur?

* **uar4**: 4 duyguda dengeli doğruluk, şans 0,25. Karşılaştırma: Aşama 1'de emotion2vec+ base hazır tahmini tüm TurEV'de
  0,79 (aynı veri, aynı etiket eşlemesi).
* **kizgin_mi_uar**: "kızgın mı, değil mi" dengeli doğruluğu, şans 0,50. Projenin asıl sorusu bu.
* **taninamayan_cevap**: TR-Omni dört kelimeden birini söylemediyse yanlış sayılır; yüksekse sorun algı değil yönerge
  izleme olabilir (`omni_algi.jsonl` içindeki `cevap` alanına bakın).
* **spearman_olculen_ofke**: TR-Omni'nin öfke sıralaması mevcut hattın ölçümüyle ne kadar örtüşüyor (n=20, kaba gösterge;
  mevcut ölçüm de hatalı olabilir, gerçek etiket değildir).
* **omni_sohbet**: `ort_sure` cevabın tamamının üretilme süresi (akış yok; canlı uygulamada ilk ses ~0,7 sn bildiriliyor).
  `cevap_sesi_duygu_dagilimi` kızgın müşteriye asistanın hangi tonda duyulduğunu gösterir. Sesleri aşağıda dinleyin.

* **omni_gecikme** (GPU başına): `ilk_ses_ort` müşteri sustuktan sonra ilk sesin çıkma süresi (ses tanıma/sessizlik algısı
  hariç). `ilk_token_ort` düşünürün ilk kelimesi, `ilk_cumle_ort` ilk cümlenin bitmesi. `konusucu_rtf_ort` < 1 ise
  seslendirme konuşmadan hızlı; > 1 ise cümleler arasında boşluk olur. Canlı görüşme için hedef: ilk ses ≤ 1,5 sn.
* **omni_enjeksiyon**: `ofkeli` koşulunda ton `şefkatli`/`sakin`e kayıyor, empati oranı artıyor ve `yok_ile_benzerlik`
  düşüyorsa model enjeksiyonu kullanıyor demektir. `sizinti` > 0 ise model notu müşteriye okuyor (ör. "öfkeniz 3,5").

* **omni_kalite** (GPU ve ayar başına): `bozuk_orani` CER > 0,3 olan cümlelerin oranı (anlaşılmayan, bozuk ses).
  `akissiz` TR-Omni'nin kendi yolu (çıta). Aynı tohumla akışlı ve akışsız yol aynı sesi üretir, yalnızca çözücü farklı:
  `akis` ≈ `akissiz` ise akış sorunsuz; `akissiz_derleme` kötüyse bozulma derlemeden; hepsi benzer oranda bozuksa
  bozulma modelin kendisinden. `bozuk_ornek` dosyaları aşağıda çalınır.
* **omni_mos**: UTMOS tahmini dinleyici puanı (1-5; iyi bir TTS ~3,5-4,3). `dusuk` 3,0'ın altındaki ses sayısı.
  `konusucu/...` satırları kulakla bozuk bulunan örnekleri içerir: puan onları düşük buluyorsa ölçüt güvenilir.
* **omni_konusucu** (GPU ve ayar başına): `rtf_ort` < 1 olmalı, yoksa akışta cümle içinde kesinti olur.
  `ilk_ses_tahmini_ort` = düşünürün ilk cümlesi + konuşucunun ilk ses parçası (hedef ≤ 1,5 sn). `akis_derleme_yarim`
  ve `_6`/`_4` daha hızlı ama kalitesi düşebilir: aşağıda aynı cümleyi ayarlar arasında dinleyip karşılaştırın.
  `akis_derleme_yarim_yuk`: aynı yarı adım, ama seslendirme sırasında arka planda düşünür de çalışır (gerçek
  görüşmedeki GPU paylaşımı). `rtf_ort` buradaki artış, gerçek kullanımda kalan payı gösterir.

**A100 karşılaştırması:** Çalışma zamanını A100 yapın, 1-3. hücreleri ve 4. hücreyi tekrar çalıştırın. Diğer
bölümler atlanır, yalnızca `omni_gecikme` ve `omni_konusucu` A100 için yeniden ölçülür; özet iki GPU'yu yan yana gösterir.

Bu sayılar ince ayarın geçmesi gereken çıtadır: `ozet.json` dosyasını saklayın.

In [ ]:
# 5) Özet ve örnek cevaplar
import json
from IPython.display import Audio, display
print(json.dumps(json.load(open(f"{CIKTI}/ozet.json")), ensure_ascii=False, indent=1))
for l in list(open(f"{CIKTI}/omni_sohbet.jsonl", encoding="utf-8"))[:4]:
    r = json.loads(l)
    print()
    print(f"Müşteri (öfke {r['musteri_ofke']}): {r['musteri']}")
    print(f"TR-Omni ({r['cevap_sesi_duygu']}): {r['cevap']}")
    display(Audio(f"{CIKTI}/sohbet_ses/{r['anahtar']}.wav"))
print()
print("--- Enjeksiyon: aynı müşteri, üç koşul ---")
enj = [json.loads(l) for l in open(f"{CIKTI}/omni_enjeksiyon.jsonl", encoding="utf-8")]
for tepki in sorted({r["tepki"] for r in enj})[:3]:
    print()
    for r in [r for r in enj if r["tepki"] == tepki]:
        print(f"{tepki} | {r['kosul']:6} [{r['ton']}]: {r['cevap']}")

print()
print("--- Konuşucu ayarları: aynı ilk cümle ---")
import os
kon = [json.loads(l) for l in open(f"{CIKTI}/omni_konusucu.jsonl", encoding="utf-8")]
for tepki in sorted({r["anahtar"].split("|")[2] for r in kon})[:3]:
    for ayar in ["akis", "akis_derleme", "akis_derleme_yarim", "akis_derleme_6", "akis_derleme_4"]:
        f = f"{CIKTI}/konusucu_ses/{ayar}_{tepki}.wav"
        if os.path.exists(f):
            print(tepki, ayar)
            display(Audio(f))

print()
print("--- Kalite: en bozuk cümleler ---")
oz = json.load(open(f"{CIKTI}/ozet.json"))
for gpu, ayarlar in oz.get("omni_kalite", {}).items():
    for ayar, v in ayarlar.items():
        print(f"{gpu} {ayar}: {v['bozuk']}/{v['cumle']} bozuk, CER ort {v['cer_ort']:.3f}")
kal = [json.loads(l) for l in open(f"{CIKTI}/omni_kalite.jsonl", encoding="utf-8")]
bozuk = sorted((c for r in kal for c in r["cumleler"] if c["cer"] > 0.3), key=lambda c: -c["cer"])
for c in bozuk[:6]:
    print()
    print(f"{c['wav']} (CER {c['cer']})")
    print(f"  metin  : {c['metin']}")
    print(f"  duyulan: {c['duyulan']}")
    display(Audio(f"{CIKTI}/{c['wav']}"))

print()
print("--- MOS (UTMOS, 1-5) ---")
for k, v in oz.get("omni_mos", {}).items():
    print(f"{k:32} n={v['n']:3}  ort {v['mos_ort']:.2f}  min {v['mos_min']:.2f}  düşük {v['dusuk']}")
mos = [json.loads(l) for l in open(f"{CIKTI}/omni_mos.jsonl", encoding="utf-8")]
print()
print("Kulakla dinlenen örnekler (konusucu_ses), puana göre:")
for r in sorted((r for r in mos if r["kaynak"] == "konusucu"), key=lambda r: r["mos"])[:12]:
    print(f"  {r['mos']:.2f}  {r['ayar']:20} {r['tepki']}")